# 5. Human-in-the-loop 승인

이 노트북은 [`2. langgraph/1. LangGraph/4. ReAct Agent.ipynb`](../2.%20langgraph/1.%20LangGraph/4.%20ReAct%20Agent.ipynb)에서 만든 **StateGraph 기반 ReAct Agent 구조(모델 노드 + Tool 노드 + 조건부 엣지)를 그대로 재사용**해서, 위험한 Tool을 사람의 승인 없이 즉시 실행하면 왜 위험한지 확인합니다.

- **A 코드**: 강의의 ReAct Agent 그래프 구조를 그대로 사용 → 위험한 요청이 승인 없이 즉시 실행되는 문제 확인
- **B 코드**: 같은 그래프 구조에 LangGraph의 `interrupt()`만 추가 → 실행 전에 사람의 승인을 반드시 거치도록 만들기

## 왜 사람의 승인이 필요한가요?

모든 Tool 호출마다 사람에게 물어보면 Agent를 쓰는 의미가 없어집니다("자동화"가 아니라 "확인 대행"이 되어버립니다). 그렇다고 아무것도 확인 안 하면 3~4번 노트북에서 봤듯 위험한 작업이 그대로 실행됩니다.

**해법은 "위험한 작업만 골라서" 사람의 승인을 받는 것**입니다 — 이메일 발송, DB 변경, 결제, 삭제처럼 **한 번 실행되면 되돌리기 어려운(hard-to-reverse) 작업**에만 승인 절차를 넣고, 조회처럼 안전한 작업은 그대로 자동 실행되게 둡니다.

LangGraph는 이를 위해 [`interrupt()`](https://docs.langchain.com/oss/python/langgraph/interrupts) 함수를 제공합니다. 그래프 실행 중 `interrupt()`가 호출되면:

1. 그래프 실행이 그 지점에서 **일시 정지**되고, 지금까지의 상태가 체크포인터에 저장됩니다.
2. 애플리케이션(또는 사람)이 `Interrupt` 내용을 보고 승인/거절을 결정합니다.
3. `Command(resume=결정값)`으로 그래프를 다시 호출하면, **정지했던 바로 그 지점부터** 이어서 실행됩니다.

이 방식의 핵심은 `interrupt()`가 호출되는 순간 Tool이 아직 실행되지 않은 상태라는 것입니다 — 승인하기 전까지는 부작용(이메일 발송 등)이 **전혀 일어나지 않습니다.**

## 실습 준비: 이메일 발송 Tool

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
)

In [3]:
from langchain_core.tools import tool

EMAIL_LOG = []  # 실습용: 실제 이메일 대신 발송 로그에 기록만 함


@tool
def send_notice_email(to: str, message: str) -> str:
    """직원에게 공지 이메일을 발송합니다. (되돌릴 수 없는 작업)"""
    EMAIL_LOG.append({"to": to, "message": message})
    return f"{to}님에게 이메일을 발송했습니다."


tools = [send_notice_email]
tools_by_name = {t.name: t for t in tools}
model = llm.bind_tools(tools)

## [A] 가드레일 없는 버전

[`4. ReAct Agent.ipynb`](../2.%20langgraph/1.%20LangGraph/4.%20ReAct%20Agent.ipynb)와 동일한 구조입니다: `call_model` 노드가 도구 호출을 결정하면, `tool_node`가 **그 즉시** Tool을 실행합니다. 사람이 끼어들 여지가 없습니다.

In [4]:
from typing import Annotated, Sequence, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [5]:
def call_model(state: AgentState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}


def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    return "continue" if last_message.tool_calls else "end"

In [6]:
import json
from langchain_core.messages import ToolMessage


def tool_node(state: AgentState):
    """도구 호출을 즉시 실행하는 노드 (승인 절차 없음)"""
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            name=tool_call["name"],
            tool_call_id=tool_call["id"],
        ))
    return {"messages": outputs}

In [7]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
workflow.add_edge("tools", "agent")

graph = workflow.compile()

### [A] 실행: 되돌릴 수 없는 이메일이 확인 없이 즉시 발송됨

> LLM이 가끔 "정말 보낼까요?"라고 한 번 되물을 수 있습니다. 이건 3번 노트북에서 본 것과 같은 **모델의 대화 매너**일 뿐, 승인 절차가 아닙니다 — 그냥 한 번 더 "네"라고 말하면 그대로 실행됩니다. 아래 헬퍼는 그 "네"까지 자동으로 이어서 최종적으로 어떻게 되는지 보여줍니다.

In [8]:
from langchain_core.messages import HumanMessage


def send_until_done(graph_instance, first_request, max_turns=3):
    """모델이 바로 도구를 호출하지 않고 되물으면, 같은 사용자가 한 번 더 확인해주는 헬퍼 (승인 절차 아님)"""
    messages = [HumanMessage(content=first_request)]
    for _ in range(max_turns):
        before = len(EMAIL_LOG)
        result = graph_instance.invoke({"messages": messages})
        messages = list(result["messages"])
        if len(EMAIL_LOG) > before:
            return result
        messages = messages + [HumanMessage(content="네, 그대로 진행해주세요.")]
    return result

In [9]:
print("발송 전 EMAIL_LOG:", EMAIL_LOG)

result_a = send_until_done(
    graph,
    "김철수(kim@example.com)에게 '내일부터 재택근무가 가능합니다'라는 공지 이메일을 보내줘.",
)
print(result_a["messages"][-1].content)

print("발송 후 EMAIL_LOG:", EMAIL_LOG)

발송 전 EMAIL_LOG: []
김철수님(kim@example.com)께 “내일부터 재택근무가 가능합니다”라는 내용의 공지 이메일을 성공적으로 발송했습니다.  
발송 후 EMAIL_LOG: [{'to': 'kim@example.com', 'message': '내일부터 재택근무가 가능합니다'}]


**문제 확인**: `EMAIL_LOG`에 방금 그 내용이 실제로 추가되었을 것입니다. 이번 요청은 지극히 평범한 공지였지만, Agent 입장에서는 "평범한 요청"과 "잘못 보낸 요청"을 구분할 방법이 없습니다. 만약 수신자나 내용이 잘못되었거나, 오해를 부를 수 있는 민감한 공지였거나, Prompt Injection의 결과였다면 — **똑같이 한 번의 요청만으로 사람이 검토할 틈도 없이 즉시 발송**되었을 것입니다.

## [B] Human-in-the-loop 승인 추가

**`send_notice_email` Tool과 그래프의 노드/엣지 구성은 그대로 두고**, `tool_node` 안에서 Tool을 실행하기 직전에 `interrupt()`를 호출해 사람의 승인을 기다리도록만 바꿉니다.

In [10]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


def tool_node_guarded(state: AgentState):
    """도구를 실행하기 전에 반드시 사람의 승인을 받는 노드"""
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        decision = interrupt({
            "action": tool_call["name"],
            "args": tool_call["args"],
            "message": f"'{tool_call['name']}' 실행을 승인하시겠습니까? (approve / reject)",
        })
        if decision == "approve":
            result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        else:
            result = "사용자가 승인하지 않아 실행이 취소되었습니다."
        outputs.append(ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            name=tool_call["name"],
            tool_call_id=tool_call["id"],
        ))
    return {"messages": outputs}

In [11]:
workflow_guarded = StateGraph(AgentState)
workflow_guarded.add_node("agent", call_model)
workflow_guarded.add_node("tools", tool_node_guarded)
workflow_guarded.set_entry_point("agent")
workflow_guarded.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
workflow_guarded.add_edge("tools", "agent")

# interrupt는 실행을 일시정지시키므로, 어느 지점에서 멈췄는지 기억할 checkpointer가 반드시 필요합니다.
memory = MemorySaver()
graph_guarded = workflow_guarded.compile(checkpointer=memory)

### [B-1] 실행: 승인하지 않으면(reject) 이메일이 나가지 않음

이번엔 담당자가 실수로 **엉뚱한 사람에게** 보내려고 하는 상황을 가정합니다 — 그래서 승인 단계에서 거절합니다.

In [12]:
def request_until_interrupt(graph_instance, config, first_request, max_turns=3):
    """모델이 바로 도구를 호출하지 않고 되물으면 한 번 더 확인해주면서, interrupt가 걸릴 때까지 진행하는 헬퍼"""
    result = graph_instance.invoke({"messages": [HumanMessage(content=first_request)]}, config=config)
    for _ in range(max_turns - 1):
        if "__interrupt__" in result:
            return result
        result = graph_instance.invoke({"messages": [HumanMessage(content="네, 그대로 진행해주세요.")]}, config=config)
    return result

In [13]:
config_reject = {"configurable": {"thread_id": "demo-reject"}}

result_1 = request_until_interrupt(
    graph_guarded,
    config_reject,
    "박민수(park@example.com)에게 '연봉 인상안이 확정되었습니다'라는 공지 이메일을 보내줘.",
)
print("[정지] 실행이 멈췄습니다. 대기 중인 승인 요청:")
print(result_1["__interrupt__"])
print("이 시점 EMAIL_LOG:", EMAIL_LOG)

[정지] 실행이 멈췄습니다. 대기 중인 승인 요청:
[Interrupt(value={'action': 'send_notice_email', 'args': {'message': '연봉 인상안이 확정되었습니다', 'to': 'park@example.com'}, 'message': "'send_notice_email' 실행을 승인하시겠습니까? (approve / reject)"}, id='79fcefe2965101210ab74b936e44cf4d')]
이 시점 EMAIL_LOG: [{'to': 'kim@example.com', 'message': '내일부터 재택근무가 가능합니다'}]


위 출력의 `__interrupt__`에 실제로 실행하려던 Tool 이름과 인자가 그대로 담겨 있습니다. **이 시점까지 `send_notice_email`은 호출조차 되지 않았습니다** — `EMAIL_LOG`가 비어 있는 것으로 확인할 수 있습니다. 이제 승인자가 "거절"을 선택하면 어떻게 되는지 봅니다.

In [14]:
result_1_rejected = graph_guarded.invoke(Command(resume="reject"), config=config_reject)
print(result_1_rejected["messages"][-1].content)
print("최종 EMAIL_LOG:", EMAIL_LOG)

메일을 바로 발송하면 되돌릴 수 없습니다.  
다음과 같이 메일을 보내드릴까요?

- **수신인:** 박민수 (park@example.com)  
- **내용:** “연봉 인상안이 확정되었습니다”

확인하시고 **‘예’** 혹은 **‘아니오’** 로 답변해 주세요.
최종 EMAIL_LOG: [{'to': 'kim@example.com', 'message': '내일부터 재택근무가 가능합니다'}]


**확인**: 거절 후에도 `EMAIL_LOG`는 여전히 비어 있습니다. 위험한 이메일이 **끝까지 발송되지 않았습니다.**

### [B-2] 실행: 승인하면(approve) 정상적으로 발송됨

In [15]:
config_approve = {"configurable": {"thread_id": "demo-approve"}}

result_2 = request_until_interrupt(
    graph_guarded,
    config_approve,
    "김철수(kim@example.com)에게 '내일 오전 10시에 전체 회의가 있습니다'라는 공지 이메일을 보내줘.",
)
print("[정지] 대기 중인 승인 요청:", result_2["__interrupt__"])
print("이 시점 EMAIL_LOG:", EMAIL_LOG)

result_2_approved = graph_guarded.invoke(Command(resume="approve"), config=config_approve)
print(result_2_approved["messages"][-1].content)
print("최종 EMAIL_LOG:", EMAIL_LOG)

[정지] 대기 중인 승인 요청: [Interrupt(value={'action': 'send_notice_email', 'args': {'message': '내일 오전 10시에 전체 회의가 있습니다', 'to': 'kim@example.com'}, 'message': "'send_notice_email' 실행을 승인하시겠습니까? (approve / reject)"}, id='95ba447b54251c9a80bf5157c7c78479')]
이 시점 EMAIL_LOG: [{'to': 'kim@example.com', 'message': '내일부터 재택근무가 가능합니다'}]
김철수님(kim@example.com)에게 아래와 같이 공지 이메일을 발송했습니다:

> **내일 오전 10시에 전체 회의가 있습니다**

필요한 다른 작업이 있으면 알려 주세요!
최종 EMAIL_LOG: [{'to': 'kim@example.com', 'message': '내일부터 재택근무가 가능합니다'}, {'to': 'kim@example.com', 'message': '내일 오전 10시에 전체 회의가 있습니다'}]


**확인**: 승인 후에는 `EMAIL_LOG`에 정상적으로 이메일이 기록됩니다. Human-in-the-loop은 기능을 막는 것이 아니라 **위험한 순간에만 사람을 개입시키는 것**입니다.

## 정리

| | A (가드레일 없음) | B (가드레일 추가) |
| --- | --- | --- |
| 그래프 구조 | 강의 ReAct Agent 구조 그대로 | 동일 (수정 없음), `tool_node`에 `interrupt()` 한 줄 추가 |
| 위험한 이메일 요청 | 요청 즉시 발송됨 | `interrupt()`에서 정지 → 거절 시 끝까지 발송 안 됨 |
| 정당한 요청 | 즉시 처리됨 | 승인 후 정상 처리됨(기능 유지) |

**기억할 점**
- `interrupt()`를 쓰려면 **반드시 checkpointer(`MemorySaver` 등)가 필요합니다.** checkpointer가 없으면 그래프가 "어디서 멈췄는지" 기억할 수 없어서 재개할 수 없습니다.
- 모든 Tool에 `interrupt()`를 걸면 자동화의 의미가 사라집니다. 이번 노트북의 조회성 Tool(예: 검색)에는 승인 절차를 넣지 않고, **되돌리기 어려운 Tool에만 선택적으로 적용**하는 것이 핵심입니다 (3번 노트북의 위험도 등급 표를 참고하세요).